# 6 · A person in the loop

A node that needs a person returns `AskPerson`. Only what reads it waits, and the run's state ends `waiting` with every question on it. `compiled.answer(state, answers)` writes the answers in and `compiled.run(state)` goes on.

In [ ]:
from typing import Annotated

from conductor import AskPerson, CompiledGraph, From, Graph, GraphNode, NodeRegistry, Param, Result, Series, Static
from conductor.widgets import Textarea

registry = NodeRegistry()


@registry.node("text-uppercase", category="text")
def uppercase(text: str) -> str:
    return text.upper()


@registry.node("text-split", category="text")
def split(text: str, separator: str = ",") -> Series[str]:
    return text.split(separator)


@registry.node("text-join", category="text")
def join(parts: Series[str], separator: str = "") -> str:
    return separator.join(parts)


Decision = Annotated[str, Result(title="Decision")]


@registry.node("approve", title="Approve", description="Asks a person to approve the proposal", category="review")
def approve(proposal: Annotated[str, Param(title="Proposal", widget=Textarea())] = "") -> Decision | AskPerson:
    """Asks a person to approve a proposal; the person's answer is the node's output."""
    return AskPerson(prompt="Approve or rewrite the proposal.", defaults={"result": proposal}, widgets={"result": Textarea()})


## One question

`shout` waits for `approve`; `other` runs in the same leg.

In [ ]:
compiled = CompiledGraph.from_graph(
    Graph(nodes=[
        approve.graph_node("approve", proposal="Ship it"),
        GraphNode(id="shout", type="text-uppercase", version=1, bindings={"text": From("approve.result")}),
        GraphNode(id="other", type="text-uppercase", version=1, bindings={"text": Static("independent")}),
    ]),
    registry,
)

state = await compiled.run()

print(state.status)
for unit in state.waiting:
    print(unit.id, unit.row, unit.prompt, [(q.name, q.default) for q in unit.questions])
print("ran in this leg:", sorted(compiled.results(state)))

The next leg:

In [ ]:
answered = compiled.answer(state, {state.waiting[0].id: {"result": "Ship it on Monday"}})

started = []
async with compiled.stream(answered) as events:
    async for event in events:
        if event.type == "node_start":
            started.append(event.node_id)
        ended = event

print(ended.state.status)
print(compiled.results(ended.state)["shout"]["result"])
print("started again:", started)

## A question per row

A node on rows asks once per row. Each waiting unit has its own id, and an answer is keyed by it. The id names its row and a hash of what the row received, so an answer meant for a row that has since moved or changed is refused, and a later leg over an edited table keeps every answer with its row. A unit left out goes on waiting.

In [ ]:
compiled = CompiledGraph.from_graph(
    Graph(nodes=[
        GraphNode(id="drafts", type="text-split", version=1, bindings={"text": Static("first draft,second draft")}),
        approve.graph_node("approve", proposal=From("drafts.result")),
        GraphNode(id="joined", type="text-join", version=1, bindings={"parts": From("approve.result"), "separator": Static(" | ")}),
    ]),
    registry,
)

state = await compiled.run()
print(state.status, [(unit.row, unit.questions[0].default) for unit in state.waiting])

answers = {unit.id: {"result": f"{unit.questions[0].default}, approved"} for unit in state.waiting}

done = await compiled.run(compiled.answer(state, answers))
print(done.status, compiled.results(done)["joined"]["result"])

From a script:

```python
paused = compiled.run_sync()
done = compiled.run_sync(compiled.answer(paused, answers))
```